In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 📢 Étape 7 : Data Storytelling & Communication (Squelette Étudiant)

Cette étape correspond au septième et dernier chapitre de data science. L'objectif est de synthétiser vos résultats pour des profils métiers ou décideurs et de proposer des visualisations interactives ou dynamiques pour valoriser vos conclusions.

### 1. Préparation de l'environnement

In [2]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))

print("Librairies prêtes pour la phase de Data Storytelling !")

Librairies prêtes pour la phase de Data Storytelling !


### 2. Synthèse métier et Storytelling

À partir de notre analyse complète du dataset Automobile (398 véhicules, 1970-1982),
voici les **5 insights majeurs** que nous avons découverts :

---

**Insight 1 — Le poids est le facteur le plus déterminant du MPG**
La corrélation entre le poids et le MPG est de **-0.832** — la plus forte du dataset.
Plus une voiture est lourde, moins elle est économique. Les constructeurs
qui veulent améliorer l'efficacité doivent prioritairement réduire le poids.

**Insight 2 — Les voitures japonaises sont les plus économes**
Les voitures japonaises atteignent en moyenne **30.45 MPG**,
contre 27.89 pour les européennes et seulement 20.08 pour les américaines.
Cet écart reflète les différences culturelles et réglementaires entre marchés.

**Insight 3 — L'efficacité énergétique a progressé de 1970 à 1982**
Le MPG moyen est passé d'environ **17 MPG en 1970** à **31 MPG en 1982**,
soit une amélioration de +82% en 12 ans — probablement liée aux chocs
pétroliers de 1973 et 1979 qui ont forcé l'industrie à innover.

**Insight 4 — Notre modèle prédit le MPG avec 91% de précision**
Le Random Forest obtient un **R² de 0.914** et une MAE de 1.61 MPG.
En pratique, si une voiture consomme réellement 25 MPG, notre modèle
prédit entre 23.4 et 26.6 MPG — précision suffisante pour des décisions d'achat.

**Insight 5 — Les voitures 4 cylindres dominent le marché économique**
Les véhicules 4 cylindres représentent la majorité des voitures
à haute efficacité (MPG élevée). Passer de 8 à 4 cylindres
réduit en moyenne la consommation de **10 MPG**.

In [3]:
# Synthèse des métriques clés
summary = {
    'Métrique': ['R² Random Forest', 'MAE Random Forest', 'RMSE Random Forest',
                 'Accuracy Classification', 'MPG moyen Japan',
                 'MPG moyen USA', 'MPG moyen Europe'],
    'Valeur': ['0.914', '1.61 MPG', '2.15 MPG',
               '88%', '30.45', '20.08', '27.89']
}

df_summary = pd.DataFrame(summary)
print("=== Synthèse des résultats du projet ===")
print(df_summary.to_string(index=False))

=== Synthèse des résultats du projet ===
               Métrique   Valeur
       R² Random Forest    0.914
      MAE Random Forest 1.61 MPG
     RMSE Random Forest 2.15 MPG
Accuracy Classification      88%
        MPG moyen Japan    30.45
          MPG moyen USA    20.08
       MPG moyen Europe    27.89


### 3. Visualisations de Communication (Plotly)

Quatre graphiques interactifs pour presenter les resultats a un public
non-technique. Chaque figure est aussi sauvegardee en `.html` dans
`data/processed/`.

In [4]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import joblib

df = pd.read_csv('../data/processed/cleaned_data_sample.csv')
df['power_to_weight'] = df['horsepower'] / df['weight']

rf = joblib.load('../models/model.pkl')

ORIGIN_MAP = {0: 'Europe', 1: 'Japon', 2: 'USA'}
COLORS     = {'USA': '#ef4444', 'Japon': '#3b82f6', 'Europe': '#22c55e'}
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Dataset : {df.shape[0]} vehicules, {df.shape[1]} variables')
print('Modele Random Forest charge.')

Dataset : 398 vehicules, 9 variables
Modele Random Forest charge.


#### Visualisation 1 — Evolution du MPG par annee (1970–1982)

MPG moyen par annee avec bande d'ecart-type, annote des deux chocs
petroliers qui ont force l'industrie a innover.

In [5]:
mpg_year = df.groupby('model_year')['mpg'].agg(['mean', 'std']).reset_index()
mpg_year.columns = ['year', 'mean', 'std']
mpg_year['year_full'] = 1900 + mpg_year['year']

fig = go.Figure()

upper = mpg_year['mean'] + mpg_year['std']
lower = mpg_year['mean'] - mpg_year['std']
fig.add_trace(go.Scatter(
    x=list(mpg_year['year_full']) + list(mpg_year['year_full'])[::-1],
    y=list(upper) + list(lower)[::-1],
    fill='toself', fillcolor='rgba(59,130,246,0.12)',
    line=dict(color='rgba(0,0,0,0)'),
    name='+/- 1 ecart-type', showlegend=True
))

fig.add_trace(go.Scatter(
    x=mpg_year['year_full'], y=mpg_year['mean'],
    mode='lines+markers', name='MPG moyen',
    line=dict(color='#3b82f6', width=3),
    marker=dict(size=8),
    hovertemplate='<b>%{x}</b><br>MPG moyen : %{y:.1f}<extra></extra>'
))

for yr, label, col in [
    (1973, '1er choc petrolier (1973)', '#ef4444'),
    (1979, '2eme choc petrolier (1979)', '#f97316')
]:
    fig.add_vline(x=yr, line_dash='dash', line_color=col, line_width=2)
    fig.add_annotation(x=yr, y=35, text=label, showarrow=False,
                       font=dict(color=col, size=11), xanchor='left', xshift=6)

fig.update_layout(
    title_text='<b>Evolution de l efficacite energetique (1970-1982)</b>',
    title_font_size=16, xaxis_title='Annee', yaxis_title='MPG moyen',
    template='plotly_white', height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

html_path = os.path.join(OUTPUT_DIR, 'story_mpg_timeline.html')
fig.write_html(html_path)
print(f'Graphique 1 sauvegarde : {html_path}')
fig.show()

Graphique 1 sauvegarde : ../data/processed\story_mpg_timeline.html


**Lecture :** le MPG stagne autour de 17–18 entre 1970 et 1973, puis deux
paliers de croissance apparaissent apres chaque choc petrolier. En 1982 le
MPG moyen atteint 31, soit **+82 % en 12 ans**.

#### Visualisation 2 — Comparaison des origines geographiques

Trois metriques cles — MPG, poids, puissance — par region du monde.

In [6]:
stats = df.groupby('origin').agg(
    mpg=('mpg', 'mean'),
    weight=('weight', 'mean'),
    hp=('horsepower', 'mean')
).reset_index()
stats['label'] = stats['origin'].map(ORIGIN_MAP)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('MPG moyen', 'Poids moyen (lbs)', 'Puissance moyenne (HP)')
)

for col_name, col_i in [('mpg', 1), ('weight', 2), ('hp', 3)]:
    bar_colors = [COLORS[lbl] for lbl in stats['label']]
    fig.add_trace(go.Bar(
        x=stats['label'], y=stats[col_name].round(1),
        marker_color=bar_colors,
        text=stats[col_name].round(1), textposition='auto',
        showlegend=False,
        hovertemplate='<b>%{x}</b> : %{y:.1f}<extra></extra>'
    ), row=1, col=col_i)

fig.update_layout(
    title_text='<b>Profil moyen par origine geographique</b>',
    title_font_size=16, template='plotly_white', height=480
)

html_path = os.path.join(OUTPUT_DIR, 'story_origin_comparison.html')
fig.write_html(html_path)
print(f'Graphique 2 sauvegarde : {html_path}')
fig.show()

Graphique 2 sauvegarde : ../data/processed\story_origin_comparison.html


**Lecture :** les voitures japonaises sont les plus legeres et les plus
economes. Les americaines sont en moyenne 60 % plus lourdes et consomment
10 MPG de plus. L'Europe se situe entre les deux avec une efficacite
superieure aux USA malgre une puissance comparable.

#### Visualisation 3 — Importance des variables (Random Forest)

Quelles caracteristiques influencent le plus le MPG predit par notre modele ?

In [7]:
feature_labels = {
    'displacement':    'Cylindree',
    'weight':          'Poids',
    'cylinders':       'Nb. cylindres',
    'horsepower':      'Puissance (HP)',
    'model_year':      'Annee',
    'acceleration':    'Acceleration',
    'power_to_weight': 'Puissance/Poids',
    'origin':          'Origine'
}

importances = pd.Series(rf.feature_importances_, index=rf.feature_names_in_)
importances.index = [feature_labels.get(f, f) for f in importances.index]
importances = importances.sort_values()

bar_colors = [
    '#ef4444' if v > 0.20 else '#3b82f6' if v > 0.10 else '#94a3b8'
    for v in importances
]

fig = go.Figure(go.Bar(
    x=importances.values, y=importances.index,
    orientation='h',
    marker_color=bar_colors,
    text=[f'{v:.1%}' for v in importances.values],
    textposition='auto',
    hovertemplate='<b>%{y}</b> : %{x:.1%}<extra></extra>'
))

fig.update_layout(
    title_text='<b>Variables les plus influentes sur le MPG (Random Forest)</b>',
    title_font_size=16,
    xaxis_title='Importance relative',
    xaxis_tickformat='.0%',
    template='plotly_white', height=450
)

html_path = os.path.join(OUTPUT_DIR, 'story_feature_importance.html')
fig.write_html(html_path)
print(f'Graphique 3 sauvegarde : {html_path}')
fig.show()

Graphique 3 sauvegarde : ../data/processed\story_feature_importance.html


**Lecture :** la cylindree (44 %) et le poids (14 %) expliquent a eux seuls
plus de la moitie de la variance du MPG. L'origine geographique contribue
a peine 0.5 % — ce sont les caracteristiques physiques qui comptent,
pas la nationalite du constructeur.

#### Visualisation 4 — Carte interactive Poids x MPG

Chaque point est un vehicule. La taille represente la puissance (HP),
la couleur l'origine geographique. Survolez pour voir les details.

In [8]:
df['origin_label']  = df['origin'].map(ORIGIN_MAP)
df['cylinders_str'] = df['cylinders'].astype(str) + ' cyl.'

fig = px.scatter(
    df, x='weight', y='mpg',
    color='origin_label', size='horsepower',
    size_max=22, symbol='cylinders_str',
    color_discrete_map=COLORS,
    labels={
        'weight':       'Poids (lbs)',
        'mpg':          'Consommation (MPG)',
        'origin_label': 'Origine',
        'cylinders_str':'Cylindres',
        'horsepower':   'Puissance (HP)'
    },
    title='<b>Carte Poids x MPG — 398 vehicules (1970-1982)</b>',
    hover_data={'horsepower': True, 'model_year': True, 'cylinders_str': True},
    template='plotly_white', height=560
)

fig.update_traces(marker=dict(opacity=0.75, line=dict(width=0.5, color='white')))
fig.update_layout(
    title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

html_path = os.path.join(OUTPUT_DIR, 'story_weight_mpg_map.html')
fig.write_html(html_path)
print(f'Graphique 4 sauvegarde : {html_path}')
fig.show()

Graphique 4 sauvegarde : ../data/processed\story_weight_mpg_map.html


**Lecture :** la relation poids–MPG est lineaire et nette. Les voitures
japonaises (bleu) se concentrent dans le quadrant haut-MPG / faible-poids.
Les americaines (rouge) dominent la zone des poids lourds (>3500 lbs).
Un vehicule sous 2500 lbs atteint presque toujours plus de 25 MPG.

### 4. Conclusion — Recommandations Metier

**Pour l'industrie automobile :**
- Reduire le poids est la priorite n°1 pour ameliorer l'efficacite
  (impact : 44 % de la variance via la cylindree liee au poids)
- Les moteurs 4 cylindres offrent le meilleur compromis efficacite / puissance
- L'evolution 1970–1982 prouve qu'une reduction de 50 % des cylindrees
  est atteignable en une decennie sous pression reglementaire

**Pour un acheteur :**
- Un vehicule sous 2500 lbs atteint presque toujours plus de 25 MPG
- L'origine geographique seule ne predit pas l'efficacite — regardez le poids
- Notre modele (R² = 0.91, MAE = 1.61 MPG) estime la consommation de tout
  vehicule avec seulement 7 caracteristiques physiques

**Limites du projet :**
- Le dataset couvre 1970–1982 ; hybrides et electriques ne sont pas representes
- 6 valeurs de puissance manquantes ont ete imputees par la mediane
- Les resultats de classification (88 %) reposent sur des seuils MPG arbitraires